In [51]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path
import requests
import time
from tqdm import tqdm
import logging
from typing import List, Dict, Optional, Tuple
import yfinance as yf
from io import StringIO

In [52]:
class MarketDateRange:
    """市場數據日期範圍控制"""
    def __init__(self, start_date: str = None, end_date: str = None):
        self.end_date = end_date if end_date else datetime.today().strftime('%Y-%m-%d')
        self.start_date = start_date
        
    @classmethod
    def last_n_days(cls, n: int) -> 'MarketDateRange':
        """
        創建最近 n 天的日期範圍
        
        Args:
            n: 天數
        
        Returns:
            MarketDateRange 實例
        """
        end_date = datetime.today()
        start_date = end_date - timedelta(days=n)
        return cls(
            start_date=start_date.strftime('%Y-%m-%d'),
            end_date=end_date.strftime('%Y-%m-%d')
        )
    
    @classmethod
    def last_month(cls) -> 'MarketDateRange':
        """創建最近一個月的日期範圍"""
        return cls.last_n_days(30)
    
    @classmethod
    def last_quarter(cls) -> 'MarketDateRange':
        """創建最近一季的日期範圍"""
        return cls.last_n_days(90)
    
    @classmethod
    def last_year(cls) -> 'MarketDateRange':
        """創建最近一年的日期範圍"""
        return cls.last_n_days(365)
    
    @classmethod
    def year_to_date(cls) -> 'MarketDateRange':
        """創建今年至今的日期範圍"""
        return cls(
            start_date=datetime.today().replace(month=1, day=1).strftime('%Y-%m-%d')
        )
    
    @property
    def date_range_str(self) -> str:
        """返回日期範圍的字符串表示"""
        return f"從 {self.start_date or '最早'} 到 {self.end_date}"

In [59]:
class TWMarketDataProcessor:
    def __init__(self, base_dir: str = "D:/Min/Python/Project/FA_Data", 
                 date_range: MarketDateRange = None):
        """
        初始化數據處理器
        
        Args:
            base_dir: 基礎數據目錄
            date_range: 數據處理的日期範圍
        """
        self.base_path = Path(base_dir)
        self.date_range = date_range or MarketDateRange()
        self.setup_directories()
        self.setup_logging()
        
        # 記錄設定的日期範圍
        self.logger.info(f"設定數據處理範圍: {self.date_range.date_range_str}")
        
    def setup_directories(self):
        """設置所需的目錄結構"""
        # 主要數據目錄
        self.daily_price_path = self.base_path / 'daily_price'
        self.meta_data_path = self.base_path / 'meta_data'
        
        # 確保目錄存在
        self.daily_price_path.mkdir(parents=True, exist_ok=True)
        self.meta_data_path.mkdir(parents=True, exist_ok=True)
        
        # 設定檔案路徑
        self.market_index_file = self.meta_data_path / 'market_index.csv'
        self.industry_index_file = self.meta_data_path / 'industry_index.csv'
        self.stock_data_file = self.meta_data_path / 'stock_data_whole.csv'
        
    def setup_logging(self):
        """設定日誌系統"""
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        
        # 移除現有的處理器
        for handler in self.logger.handlers[:]:
            self.logger.removeHandler(handler)
        
        # 檔案處理器
        file_handler = logging.FileHandler(
            self.meta_data_path / 'market_data_process.log',
            encoding='utf-8'
        )
        file_handler.setFormatter(
            logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        )
        
        # 控制台處理器
        console_handler = logging.StreamHandler()
        console_handler.setFormatter(
            logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        )
        
        self.logger.addHandler(file_handler)
        self.logger.addHandler(console_handler)

    def get_latest_date(self, file_path: Path, date_column: str = '日期') -> Optional[str]:
        """獲取指定文件的最新日期"""
        if not file_path.exists():
            return None
            
        try:
            df = pd.read_csv(file_path)
            if df.empty:
                return None
            return df[date_column].max()
        except Exception as e:
            self.logger.error(f"讀取{file_path}的最新日期時發生錯誤: {str(e)}")
            return None

    def get_daily_stock_data(self, date_str: str) -> Optional[pd.DataFrame]:
        """從TWSE獲取每日股票資料"""
        url = f'http://www.twse.com.tw/exchangeReport/MI_INDEX?response=csv&date={date_str}&type=ALL'
        
        try:
            response = requests.get(url)
            if response.status_code != 200:
                self.logger.warning(f"無法獲取 {date_str} 的數據: HTTP {response.status_code}")
                return None

            # 解析CSV數據
            lines = [line.strip() for line in response.text.split('\n') if line.strip()]
            data_lines = []
            start_parsing = False
            
            for line in lines:
                if '證券代號' in line:
                    start_parsing = True
                    data_lines.append(line)
                elif start_parsing and not line.startswith('='):
                    data_lines.append(line)

            if data_lines:
                df = pd.read_csv(StringIO('\n'.join(data_lines)), thousands=',')
                df = df.loc[:, ~df.columns.str.contains("Unnamed")]
                
                # 標準化欄位名稱
                expected_columns = [
                    '證券代號', '證券名稱', '成交股數', '成交筆數', '成交金額',
                    '開盤價', '最高價', '最低價', '收盤價', '漲跌(+/-)',
                    '漲跌價差', '最後揭示買價', '最後揭示買量',
                    '最後揭示賣價', '最後揭示賣量', '本益比'
                ]
                
                # 確保所有必要欄位存在
                for col in expected_columns:
                    if col not in df.columns:
                        df[col] = None
                
                # 數據類型轉換
                df['證券代號'] = df['證券代號'].astype(str)
                numeric_columns = [
                    '成交股數', '成交筆數', '成交金額', '開盤價', '最高價',
                    '最低價', '收盤價', '漲跌價差', '最後揭示買價',
                    '最後揭示買量', '最後揭示賣價', '最後揭示賣量', '本益比'
                ]
                
                # 移除逗號並轉換為數值
                for col in numeric_columns:
                    if col in df.columns:
                        df[col] = pd.to_numeric(
                            df[col].str.replace(',', ''),
                            errors='coerce'
                        )
                
                return df
            return None

        except Exception as e:
            self.logger.error(f"處理 {date_str} 的股票數據時發生錯誤: {str(e)}")
            return None

    def process_daily_stock_data(self, start_date: str = None, end_date: str = None) -> Optional[pd.DataFrame]:
        """處理每日股票數據"""
        try:
            # 確定日期範圍
            if start_date is None:
                latest_date = self.get_latest_date(self.stock_data_file)
                if latest_date:
                    start_date = (datetime.strptime(latest_date, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
                else:
                    start_date = '2014-01-01'  # 預設起始日期
            
            if end_date is None:
                end_date = datetime.today().strftime('%Y-%m-%d')

            start = datetime.strptime(start_date, '%Y-%m-%d')
            end = datetime.strptime(end_date, '%Y-%m-%d')
            
            self.logger.info(f"開始處理從 {start_date} 到 {end_date} 的每日股票數據")
            
            # 讀取現有數據
            existing_df = pd.DataFrame()
            if self.stock_data_file.exists():
                existing_df = pd.read_csv(self.stock_data_file, low_memory=False)
                existing_df['證券代號'] = existing_df['證券代號'].astype(str)
                self.logger.info(f"已讀取現有數據，共 {len(existing_df)} 筆記錄")

            # 收集新數據
            new_data_frames = []
            date_range = [start + timedelta(days=x) for x in range((end-start).days + 1)]
            
            for date in tqdm(date_range, desc="處理每日股票數據進度"):
                date_str = date.strftime('%Y%m%d')
                file_path = self.daily_price_path / f'{date_str}.csv'
                
                if file_path.exists():
                    self.logger.debug(f"讀取現有文件 {date_str}")
                    daily_data = pd.read_csv(file_path)
                else:
                    daily_data = self.get_daily_stock_data(date_str)
                    if daily_data is not None:
                        daily_data.to_csv(file_path, index=False, encoding='utf-8-sig')
                        time.sleep(3)  # 避免過度頻繁請求
                
                if daily_data is not None:
                    daily_data['證券代號'] = daily_data['證券代號'].astype(str)
                    daily_data['日期'] = date.strftime('%Y-%m-%d')
                    new_data_frames.append(daily_data)

            if not new_data_frames:
                self.logger.info("沒有新的股票數據需要處理")
                return existing_df if not existing_df.empty else None

            # 合併新舊數據
            new_df = pd.concat(new_data_frames, ignore_index=True)
            if not existing_df.empty:
                df = pd.concat([existing_df, new_df], ignore_index=True)
                df = df.drop_duplicates(subset=['證券代號', '日期'], keep='last')
            else:
                df = new_df

            # 排序和保存
            df = df.sort_values(['證券代號', '日期'])
            df.to_csv(self.stock_data_file, index=False, encoding='utf-8-sig')
            
            self._generate_stock_report(df)
            return df

        except Exception as e:
            self.logger.error(f"處理每日股票數據時發生錯誤: {str(e)}")
            raise

    def update_market_index(self, start_date: str = None, end_date: str = None) -> Optional[pd.DataFrame]:
        """
        更新市場指數數據
        
        Args:
            start_date: 開始日期 (YYYY-MM-DD)
            end_date: 結束日期 (YYYY-MM-DD)
        
        Returns:
            Optional[pd.DataFrame]: 更新後的市場指數數據
        """
        try:
            self.logger.info("開始更新市場指數數據")
            
            # 如果沒有指定起始日期，檢查現有數據的最新日期
            if start_date is None:
                if self.market_index_file.exists():
                    latest_date = self.get_latest_date(self.market_index_file, 'Date')
                    if latest_date:
                        start_date = (
                            datetime.strptime(latest_date, '%Y-%m-%d') + 
                            timedelta(days=1)
                        ).strftime('%Y-%m-%d')
                    else:
                        start_date = "2014-01-01"  # 預設起始日期
                else:
                    start_date = "2014-01-01"  # 預設起始日期
            
            # 如果沒有指定結束日期，使用今天
            if end_date is None:
                end_date = datetime.today().strftime('%Y-%m-%d')
            
            # 使用yfinance獲取TAIEX數據
            self.logger.info(f"從 {start_date} 到 {end_date} 獲取TAIEX數據")
            taiex_data = yf.download("^TWII", start=start_date, end=end_date)
            
            if taiex_data.empty:
                self.logger.warning("未獲取到新的TAIEX數據")
                return None
            
            # 如果存在舊數據，進行合併
            if self.market_index_file.exists():
                old_data = pd.read_csv(self.market_index_file, index_col='Date', parse_dates=True)
                # 合併數據並去除重複
                taiex_data = pd.concat([old_data, taiex_data])
                taiex_data = taiex_data[~taiex_data.index.duplicated(keep='last')]
                taiex_data = taiex_data.sort_index()
            
            # 保存數據
            taiex_data.to_csv(self.market_index_file, encoding='utf-8-sig')
            self.logger.info(f"已更新TAIEX數據，共 {len(taiex_data)} 筆記錄")
            
            return taiex_data
            
        except Exception as e:
            self.logger.error(f"更新市場指數時發生錯誤: {str(e)}")
            raise

    def extract_index_data_for_date(self, date_str: str) -> Optional[List[Dict]]:
        """擷取特定日期的產業指數資料"""
        url = f'http://www.twse.com.tw/exchangeReport/MI_INDEX?response=csv&date={date_str}&type=ALL'
        
        try:
            response = requests.get(url)
            if response.status_code != 200:
                return None

            lines = response.text.split('\n')
            index_data = []
            start_parsing = False
            
            for line in lines:
                if '類型' in line:
                    break
                    
                if start_parsing:
                    try:
                        parts = line.replace('"', '').strip().split(',')
                        if len(parts) >= 5 and any(x in parts[0] for x in ['指數', '發行量加權']):
                            index_data.append({
                                '指數名稱': parts[0].strip(),
                                '收盤指數': float(parts[1].replace(',', '')),
                                '漲跌': parts[2],
                                '漲跌點數': float(parts[3].replace(',', '')),
                                '漲跌百分比': float(parts[4].replace(',', '')),
                                '日期': datetime.strptime(date_str, '%Y%m%d').strftime('%Y-%m-%d')
                            })
                    except (ValueError, IndexError):
                        continue
                
                if '收盤指數' in line:
                    start_parsing = True
                    
            return index_data
            
        except Exception as e:
            self.logger.error(f"處理 {date_str} 的指數數據時發生錯誤: {str(e)}")
            return None

    def process_industry_index_data(self, start_date: str = None, end_date: str = None) -> Optional[pd.DataFrame]:
        """處理產業指數數據，優化版本"""
        try:
            # 確定日期範圍
            if start_date is None:
                latest_date = self.get_latest_date(self.industry_index_file)
                if latest_date:
                    start_date = (datetime.strptime(latest_date, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
                else:
                    start_date = '2014-04-07'  # 預設起始日期
            
            if end_date is None:
                end_date = datetime.today().strftime('%Y-%m-%d')
    
            start = datetime.strptime(start_date, '%Y-%m-%d')
            end = datetime.strptime(end_date, '%Y-%m-%d')
            
            self.logger.info(f"開始處理從 {start_date} 到 {end_date} 的產業指數數據")
            
            # 讀取現有數據
            existing_df = pd.DataFrame()
            existing_dates = set()
            if self.industry_index_file.exists():
                existing_df = pd.read_csv(self.industry_index_file)
                existing_dates = set(existing_df['日期'].unique())
                self.logger.info(f"已讀取現有數據，共 {len(existing_df)} 筆記錄")
    
            # 生成需要處理的日期清單
            date_range = [start + timedelta(days=x) for x in range((end-start).days + 1)]
            dates_to_process = []
            
            for date in date_range:
                date_str = date.strftime('%Y-%m-%d')
                if date_str not in existing_dates:
                    dates_to_process.append(date)
    
            if not dates_to_process:
                self.logger.info("所有日期的數據都已存在，無需更新")
                return existing_df
    
            self.logger.info(f"需要處理 {len(dates_to_process)} 天的數據")
    
            # 收集新數據
            new_data = []
            for date in tqdm(dates_to_process, desc="處理產業指數數據進度"):
                date_str = date.strftime('%Y%m%d')
                index_data = self.extract_index_data_for_date(date_str)
                
                if index_data:
                    new_data.extend(index_data)
                time.sleep(3)  # 保留延遲以避免請求過於頻繁
    
            if not new_data:
                self.logger.info("沒有新的產業指數數據需要處理")
                return existing_df if not existing_df.empty else None
    
            # 合併新舊數據
            new_df = pd.DataFrame(new_data)
            if not existing_df.empty:
                df = pd.concat([existing_df, new_df], ignore_index=True)
                df = df.drop_duplicates(subset=['日期', '指數名稱'], keep='last')
            else:
                df = new_df
    
            # 排序和保存
            df = df.sort_values(['指數名稱', '日期'])
            df.to_csv(self.industry_index_file, index=False, encoding='utf-8-sig')
            
            self._generate_index_report(df)
            return df
    
        except Exception as e:
            self.logger.error(f"處理產業指數數據時發生錯誤: {str(e)}")
            raise
    
    def _generate_index_report(self, df: pd.DataFrame):
        """生成指數數據報告，增加處理日期的統計"""
        self.logger.info("\n=== 指數數據報告 ===")
        self.logger.info(f"總記錄數: {len(df):,d}")
        self.logger.info(f"指數數量: {len(df['指數名稱'].unique()):,d}")
        self.logger.info(f"日期範圍: {df['日期'].min()} 到 {df['日期'].max()}")
        self.logger.info(f"總交易日數: {len(df['日期'].unique()):,d}")
        
        # 顯示最近更新的日期
        recent_dates = sorted(df['日期'].unique())[-5:]
        self.logger.info(f"最近的5個交易日: {', '.join(recent_dates)}")
    
        def _generate_index_report(self, df: pd.DataFrame):
            """生成指數數據報告"""
            self.logger.info("\n=== 指數數據報告 ===")
            self.logger.info(f"總記錄數: {len(df):,d}")
            self.logger.info(f"指數數量: {len(df['指數名稱'].unique()):,d}")
            self.logger.info(f"日期範圍: {df['日期'].min()} 到 {df['日期'].max()}")
            
            try:
                # 最新日期的指數漲跌情況
                latest_date = df['日期'].max()
                latest_data = df[df['日期'] == latest_date]
                
                self.logger.info(f"\n最新交易日 ({latest_date}) 指數表現:")
                for _, row in latest_data.iterrows():
                    self.logger.info(
                        f"  - {row['指數名稱']}: {row['收盤指數']:,.2f} "
                        f"({row['漲跌']} {row['漲跌點數']:,.2f} / {row['漲跌百分比']:,.2f}%)"
                    )
                
                # 計算期間漲幅最大的指數
                df_period = df.sort_values('日期')
                index_performance = []
                
                for name in df['指數名稱'].unique():
                    index_data = df[df['指數名稱'] == name]
                    if len(index_data) >= 2:
                        first_value = index_data.iloc[0]['收盤指數']
                        last_value = index_data.iloc[-1]['收盤指數']
                        change_pct = ((last_value - first_value) / first_value) * 100
                        index_performance.append({
                            '指數名稱': name,
                            '漲跌幅': change_pct
                        })
                
                if index_performance:
                    performance_df = pd.DataFrame(index_performance)
                    top_performers = performance_df.nlargest(5, '漲跌幅')
                    
                    self.logger.info("\n期間漲幅最大的指數:")
                    for _, row in top_performers.iterrows():
                        self.logger.info(
                            f"  - {row['指數名稱']}: {row['漲跌幅']:,.2f}%"
                        )
                        
            except Exception as e:
                self.logger.error(f"生成指數報告時發生錯誤: {str(e)}")

    def _generate_stock_report(self, df: pd.DataFrame):
        """生成加強版股票數據報告，區分期間統計和當日統計"""
        try:
            self.logger.info("\n=== 股票數據報告 ===")
            self.logger.info(f"總記錄數: {len(df):,d}")
            self.logger.info(f"股票數量: {len(df['證券代號'].unique()):,d}")
            self.logger.info(f"日期範圍: {df['日期'].min()} 到 {df['日期'].max()}")
            self.logger.info(f"總交易日數: {len(df['日期'].unique()):,d}")
            
            # 期間統計
            self.logger.info(f"\n=== 期間統計 ({df['日期'].min()} 到 {df['日期'].max()}) ===")
            
            # 計算期間總成交量
            period_volume = df.groupby(['證券代號', '證券名稱'])['成交股數'].sum().sort_values(ascending=False)
            self.logger.info("\n期間成交量最大的5支股票:")
            for idx, volume in period_volume.head().items():
                self.logger.info(
                    f"  - {idx[0]} ({idx[1]}): {volume:,d} 股"
                )
            
            # 計算期間總成交金額
            period_value = df.groupby(['證券代號', '證券名稱'])['成交金額'].sum().sort_values(ascending=False)
            self.logger.info("\n期間成交金額最大的5支股票:")
            for idx, value in period_value.head().items():
                self.logger.info(
                    f"  - {idx[0]} ({idx[1]}): NT$ {value:,.0f}"
                )
            
            # 最新交易日統計
            latest_date = df['日期'].max()
            latest_data = df[df['日期'] == latest_date]
            
            self.logger.info(f"\n=== 最新交易日統計 ({latest_date}) ===")
            self.logger.info(f"當日成交股票數: {len(latest_data):,d}")
            
            # 當日成交量排行
            daily_volume = latest_data.nlargest(5, '成交股數')
            self.logger.info("\n當日成交量最大的5支股票:")
            for _, row in daily_volume.iterrows():
                self.logger.info(
                    f"  - {row['證券代號']} ({row['證券名稱']}): {row['成交股數']:,d} 股"
                )
            
            # 當日成交金額排行
            daily_value = latest_data.nlargest(5, '成交金額')
            self.logger.info("\n當日成交金額最大的5支股票:")
            for _, row in daily_value.iterrows():
                self.logger.info(
                    f"  - {row['證券代號']} ({row['證券名稱']}): NT$ {row['成交金額']:,.0f}"
                )
            
            # 當日漲跌統計
            if '漲跌(+/-)' in latest_data.columns:
                up_count = len(latest_data[latest_data['漲跌(+/-)'] == '+'])
                down_count = len(latest_data[latest_data['漲跌(+/-)'] == '-'])
                unchanged = len(latest_data) - up_count - down_count
                
                self.logger.info(f"\n當日漲跌家數:")
                self.logger.info(f"  - 上漲: {up_count:,d}")
                self.logger.info(f"  - 下跌: {down_count:,d}")
                self.logger.info(f"  - 持平: {unchanged:,d}")
    
            # 交易日期列表
            recent_dates = sorted(df['日期'].unique())[-5:]
            self.logger.info(f"\n最近的5個交易日: {', '.join(recent_dates)}")
            
        except Exception as e:
            self.logger.error(f"生成股票報告時發生錯誤: {str(e)}")
    
    def process_all(self):
        """執行完整的數據處理流程"""
        self.logger.info(f"開始執行完整數據更新流程: {self.date_range.date_range_str}")
        
        try:
            # 1. 更新大盤指數
            self.logger.info("\n=== 更新大盤指數 ===")
            taiex_data = self.update_market_index(
                start_date=self.date_range.start_date,
                end_date=self.date_range.end_date
            )
            
            # 2. 更新產業指數
            self.logger.info("\n=== 更新產業指數 ===")
            industry_data = self.process_industry_index_data(
                start_date=self.date_range.start_date,
                end_date=self.date_range.end_date
            )
            
            # 3. 更新個股數據
            self.logger.info("\n=== 更新個股數據 ===")
            stock_data = self.process_daily_stock_data(
                start_date=self.date_range.start_date,
                end_date=self.date_range.end_date
            )
            
            # 生成更新報告
            self._generate_update_summary()
            
            return {
                'taiex_data': taiex_data,
                'industry_data': industry_data,
                'stock_data': stock_data
            }
            
        except Exception as e:
            self.logger.error(f"更新過程中發生錯誤: {str(e)}")
            raise

    def _generate_update_summary(self):
        """生成數據更新摘要報告（優化版）"""
        self.logger.info("\n=== 數據更新摘要 ===")
        
        # 檢查各個數據文件
        files_status = {
            '大盤指數': self.market_index_file,
            '產業指數': self.industry_index_file,
            '個股數據': self.stock_data_file
        }
        
        for name, file_path in files_status.items():
            if file_path.exists():
                try:
                    if name == '大盤指數':
                        df = pd.read_csv(file_path, low_memory=False)
                        self.logger.info(f"{name}:")
                        self.logger.info(f"  - 資料筆數: {len(df):,d}")
                        self.logger.info(f"  - 日期範圍: {df['Date'].min()} 到 {df['Date'].max()}")
                    else:
                        # 為個股數據指定數據類型
                        if name == '個股數據':
                            dtype_dict = {
                                '證券代號': str,
                                '證券名稱': str,
                                '成交股數': float,
                                '成交筆數': float,
                                '成交金額': float,
                                '開盤價': float,
                                '最高價': float,
                                '最低價': float,
                                '收盤價': float,
                                '漲跌(+/-)': str
                            }
                            df = pd.read_csv(file_path, dtype=dtype_dict, low_memory=False)
                        else:
                            df = pd.read_csv(file_path, low_memory=False)
                        
                        self.logger.info(f"{name}:")
                        self.logger.info(f"  - 資料筆數: {len(df):,d}")
                        self.logger.info(f"  - 日期範圍: {df['日期'].min()} 到 {df['日期'].max()}")
                        if name == '產業指數':
                            self.logger.info(f"  - 指數數量: {len(df['指數名稱'].unique())}")
                        elif name == '個股數據':
                            self.logger.info(f"  - 股票數量: {len(df['證券代號'].unique())}")
                except Exception as e:
                    self.logger.error(f"讀取 {name} 數據時發生錯誤: {str(e)}")
            else:
                self.logger.warning(f"{name} 數據文件不存在")

    def show_data_structure(self):
        """顯示所有數據文件的基本結構"""
        self.logger.info("\n=== 數據文件結構總覽 ===")
        
        files_info = {
            '大盤指數': {
                'path': self.market_index_file,
                'description': '台灣加權指數每日交易數據'
            },
            '產業指數': {
                'path': self.industry_index_file,
                'description': '各產業指數每日數據'
            },
            '個股數據': {
                'path': self.stock_data_file,
                'description': '個股每日交易詳細數據'
            }
        }
        
        for name, info in files_info.items():
            self.logger.info(f"\n{'-'*50}")
            self.logger.info(f"【{name}】")
            self.logger.info(f"說明: {info['description']}")
            
            if not info['path'].exists():
                self.logger.warning(f"檔案不存在: {info['path']}")
                continue
                
            try:
                # 讀取檔案（只讀取前100行來分析結構）
                df = pd.read_csv(info['path'], low_memory=False, nrows=100)
                
                # 基本信息
                self.logger.info(f"欄位數量: {len(df.columns)}")
                
                # 欄位資訊表格
                self.logger.info("\n欄位明細:")
                self.logger.info(f"{'欄位名稱':<20} {'數據類型':<15} {'值示例':<30}")
                self.logger.info("-" * 65)
                
                for col in df.columns:
                    # 獲取值示例
                    sample_value = df[col].dropna().iloc[0] if not df[col].empty else 'N/A'
                    # 格式化輸出，確保對齊
                    self.logger.info(
                        f"{str(col):<20} {str(df[col].dtype):<15} {str(sample_value):<30}"
                    )
                    
            except Exception as e:
                self.logger.error(f"分析 {name} 時發生錯誤: {str(e)}")

In [54]:
# processor.update_market_index()  # 更新大盤指數
# processor.process_industry_index_data()  # 更新產業指數
# processor.process_daily_stock_data()  # 更新個股數據

# 處理日期範圍(有end_date就會給範圍,沒有則預設為今天
# date_range = MarketDateRange(start_date='2023-01-01',end_date='2023-12-31')
# date_range = MarketDateRange(start_date='2024-01-01')
# processor = TWMarketDataProcessor(date_range=date_range)

#function
# 最近一個月的數據
# processor = TWMarketDataProcessor(date_range=MarketDateRange.last_month())
# # 最近90天（一季）的數據
# processor = TWMarketDataProcessor(date_range=MarketDateRange.last_quarter())
# # 最近一年的數據
# processor = TWMarketDataProcessor(date_range=MarketDateRange.last_year())
# # 今年至今的數據
# processor = TWMarketDataProcessor(date_range=MarketDateRange.year_to_date())
# 自定義天數
# processor = TWMarketDataProcessor(date_range=MarketDateRange.last_n_days(45))
#顯示數據結構
# processor.show_data_structure()  # 在更新數據之前或之後都可以調用

In [61]:
if __name__ == "__main__":
    processor = TWMarketDataProcessor(date_range=MarketDateRange.last_n_days(45))
    processor.process_all()

2024-10-29 14:38:45,129 - INFO - 設定數據處理範圍: 從 2024-09-14 到 2024-10-29
2024-10-29 14:38:45,129 - INFO - 開始執行完整數據更新流程: 從 2024-09-14 到 2024-10-29
2024-10-29 14:38:45,130 - INFO - 
=== 更新大盤指數 ===
2024-10-29 14:38:45,131 - INFO - 開始更新市場指數數據
2024-10-29 14:38:45,132 - INFO - 從 2024-09-14 到 2024-10-29 獲取TAIEX數據
[*********************100%%**********************]  1 of 1 completed
2024-10-29 14:38:45,183 - INFO - 已更新TAIEX數據，共 2633 筆記錄
2024-10-29 14:38:45,184 - INFO - 
=== 更新產業指數 ===
2024-10-29 14:38:45,185 - INFO - 開始處理從 2024-09-14 到 2024-10-29 的產業指數數據
2024-10-29 14:38:45,345 - INFO - 已讀取現有數據，共 149863 筆記錄
2024-10-29 14:38:45,346 - INFO - 需要處理 18 天的數據
處理產業指數數據進度: 100%|████████████████████████████████████████████████████████████| 18/18 [01:19<00:00,  4.40s/it]
2024-10-29 14:40:04,502 - INFO - 沒有新的產業指數數據需要處理
2024-10-29 14:40:04,503 - INFO - 
=== 更新個股數據 ===
2024-10-29 14:40:04,504 - INFO - 開始處理從 2024-09-14 到 2024-10-29 的每日股票數據
2024-10-29 14:40:08,351 - INFO - 已讀取現有數據，共 2466511 筆記錄
處理每日股票數據進度: 100%|██

In [60]:
if __name__ == "__main__":
    processor = TWMarketDataProcessor()
    processor.show_data_structure()  # 在更新數據之前或之後都可以調用

2024-10-29 14:37:45,721 - INFO - 設定數據處理範圍: 從 最早 到 2024-10-29
2024-10-29 14:37:45,722 - INFO - 
=== 數據文件結構總覽 ===
2024-10-29 14:37:45,722 - INFO - 
--------------------------------------------------
2024-10-29 14:37:45,723 - INFO - 【大盤指數】
2024-10-29 14:37:45,723 - INFO - 說明: 台灣加權指數每日交易數據
2024-10-29 14:37:45,730 - INFO - 欄位數量: 7
2024-10-29 14:37:45,730 - INFO - 
欄位明細:
2024-10-29 14:37:45,731 - INFO - 欄位名稱                 數據類型            值示例                           
2024-10-29 14:37:45,732 - INFO - -----------------------------------------------------------------
2024-10-29 14:37:45,733 - INFO - Date                 object          2014-01-02                    
2024-10-29 14:37:45,733 - INFO - Open                 float64         8618.599609375                
2024-10-29 14:37:45,734 - INFO - High                 float64         8632.8095703125               
2024-10-29 14:37:45,735 - INFO - Low                  float64         8587.5400390625               
2024-10-29 14:37:45,736 - IN